In [ ]:
import logging
import json
import difflib
import httpx
from typing import List, Dict, Optional, Tuple
from urllib.parse import urljoin

from bs4 import BeautifulSoup

from config import Config

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)


class PlantInfoFetcher:
    """Helper for identifying plants and retrieving care information."""

    def __init__(self, timeout: float = 10.0) -> None:
        self.client = httpx.Client(timeout=timeout)
        logger.info("Initialized PlantInfoFetcher with timeout=%s", timeout)

    def identify_with_plantnet(
        self,
        files: List[tuple[str, str]],
        data: Dict = None,
    ) -> Dict[str, object]:
        api_url = f"{Config.PLANTNET_API}{Config.PROJECT}?api-key={Config.PLANTNET_KEY}"
        logger.debug("Sending image data to PlantNet API at %s", api_url)
        try:
            response = httpx.post(api_url, data=data, files=files, timeout=10.0)
            response.raise_for_status()
            logger.debug("Received response from PlantNet API")
            payload = response.json()
            results = payload.get("results", [])
            if not results:
                logger.warning("No results from PlantNet API")
                raise ValueError("No results returned from PlantNet API")
            top = results[0]
            species = top.get("species", {})
            return {
                "score": top.get("score"),
                "common_names": species.get("commonNames", []),
                "scientific_name": species.get("scientificName"),
            }
        except httpx.RequestError as e:
            logger.exception("Request to PlantNet API failed")
            raise
        except (KeyError, ValueError, TypeError) as e:
            logger.error("Error parsing PlantNet response: %s", e)
            raise

    def _build_dict(self, sections: List[Dict[str, object]]) -> Dict[str, str]:
        result: Dict[str, str] = {}
        logger.debug("Building dictionary from section data")
        for item in sections:
            item_id = item.get("id", "<unknown>")
            try:
                t = item["type"]
                desc = item["description"]
                if not isinstance(t, str) or not isinstance(desc, str):
                    raise TypeError("Invalid section entry types")
                if t in result:
                    raise ValueError(f"Duplicate section type '{t}'")
                result[t] = desc
            except KeyError as exc:
                logger.warning("Missing key %s in section %s", exc, item_id)
            except Exception as exc:
                logger.exception("Error processing section %s", item_id)
        return result

    def fetch_care_from_perenual(self, plant_name: str) -> Dict[str, str]:
        logger.info("Fetching care data for plant: %s", plant_name)
        url = (
            f"{Config.PERENUAL_API}{Config.PERENUAL_ENDPOINT}?key={Config.PERENUAL_KEY}&q={plant_name}"
        )
        try:
            resp = self.client.get(url)
            resp.raise_for_status()
            data = resp.json()
            logger.debug("Fetched care data from Perenual API")
        except httpx.HTTPError as exc:
            logger.error("Failed fetching care guide for %s: %s", plant_name, exc)
            return {}
        except json.JSONDecodeError as exc:
            logger.error("Invalid JSON for %s: %s", plant_name, exc)
            return {}
        try:
            top = data["data"][0]
            sections = top.get("section", [])
            return self._build_dict(sections)
        except (KeyError, IndexError, TypeError) as exc:
            logger.error("Unexpected response structure for %s: %s", plant_name, exc)
            return {}

    def scrape_perenual_search(self, query: str) -> List[Tuple[str, str]]:
        logger.info("Scraping search results for: %s", query)
        base = "https://perenual.com/plant-species-database-search-finder"
        headers = {"User-Agent": "Mozilla/5.0"}
        try:
            resp = self.client.get(base, params={"search": query}, headers=headers)
            resp.raise_for_status()
            logger.debug("Received HTML response for search query")
        except httpx.HTTPError as exc:
            logger.error("Error searching Perenual: %s", exc)
            return []
        soup = BeautifulSoup(resp.text, "html.parser")
        results: List[Tuple[str, str]] = []
        for a in soup.select(".search-container-box a[href]"):
            url = urljoin(base, a.get("href"))
            name = a.get_text(strip=True)
            results.append((name, url))
        logger.debug("Extracted %d search results", len(results))
        return results

    def scrape_perenual_page(self, url: str) -> Dict[str, str]:
        logger.info("Scraping care page at: %s", url)
        try:
            resp = self.client.get(url)
            resp.raise_for_status()
            logger.debug("Received HTML content for care page")
        except httpx.HTTPError as exc:
            logger.error("Failed to fetch Perenual page: %s", exc)
            return {}
        soup = BeautifulSoup(resp.text, "html.parser")
        paras = soup.select("p.whitespace-pre-wrap")
        texts = [p.get_text(strip=True) for p in paras[:3]]
        keys = ["watering", "sunlight", "pruning"]
        return dict(zip(keys, texts))

    def get_plant_info(self, image_files: list, organs: list) -> Dict[str, object]:
        logger.info("Identifying plant and retrieving care information")
        files = []
        for image_file in image_files:
            logger.debug("Reading image file: %s", image_file)
            image_data = open(image_file, 'rb')
            files.append(('images', image_data))

        if organs is None:
            organs = ["auto" for _ in image_files]

        data = {'organs': organs}
        identified = self.identify_with_plantnet(data=data, files=files)

        names = [identified.get("scientific_name", "")] + identified.get("common_names", [])
        logger.debug("Attempting to fetch care data for names: %s", names)
        care = self.fetch_care_from_perenual(identified["scientific_name"])
        if not care:
            for name in identified.get("common_names", []):
                care = self.fetch_care_from_perenual(name)
                if care:
                    break
        if not care:
            logger.info("Fallback to scraping perenual.com for care info")
            search_results = self.scrape_perenual_search(identified["scientific_name"])
            if search_results:
                ranked = self._rank_by_similarity(search_results, names)
                for _name, url in ranked:
                    care = self.scrape_perenual_page(url)
                    if care:
                        break
        identified.update(care)
        logger.info("Completed identification and care retrieval")
        return identified

    def _rank_by_similarity(self, results: List[Tuple[str, str]], names: List[str]) -> List[Tuple[str, str]]:
        logger.debug("Ranking search results by similarity")
        def score(title: str) -> float:
            title_lower = title.lower()
            best = 0.0
            for n in names:
                if not n:
                    continue
                best = max(best, difflib.SequenceMatcher(None, title_lower, n.lower()).ratio())
            return best
        ranked = sorted(results, key=lambda t: score(t[0]), reverse=True)
        logger.debug("Top ranked results: %s", ranked[:3])
        return ranked


In [7]:
fetcher = PlantInfoFetcher()
image_files = ["./../../plant_id_test/test_photos/camellia.jpg"]
organs = ["auto"]
files = []
for image_file in image_files:
    image_data = open(image_file, 'rb')
    files.append(('images', image_data))

if organs is None:
    organs = ["auto" for _ in image_files]

data = {'organs': organs}
info = fetcher.identify_with_plantnet(files=files,data=data)
info

2025-07-26 20:58:16 [INFO] __main__: Initialized PlantInfoFetcher with timeout=10.0


2025-07-26 20:58:19 [INFO] httpx: HTTP Request: POST https://my-api.plantnet.org/v2/identify/all?api-key=2b10uztmWSXBQhAsctxZTu "HTTP/1.1 200 OK"


{'score': 0.36081,
 'common_names': ['Camellia', 'Sasanqua camellia', 'Christmas camellia'],
 'scientific_name': 'Camellia sasanqua Thunb.'}

In [8]:
guide = fetcher.fetch_care_from_perenual("Camellia")
guide

2025-07-26 20:58:19 [INFO] __main__: Fetching care data for plant: Camellia
2025-07-26 20:58:21 [INFO] httpx: HTTP Request: GET https://perenual.com/api/species-care-guide-list?key=sk-BY6T687a9e958639511466&q=Camellia "HTTP/1.1 200 OK"


{'watering': 'Camellias (Camellia crapnelliana) enjoy an ample amount of water but should never be overwatered. Aim to water your camellia once or twice a week, ultimately depending on how warm or dry your environment is. A good way to tell when it’s time to water is by allowing your soil to become somewhat dry before watering again, but always avoid allowing the soil to dry completely. Never water your plant’s leaves or leave standing water. Camellias also benefit from misting, which can be done every couple of days or as needed.',
 'sunlight': 'Camellia crapnelliana is best grown in a spot that receives full to partial sun, such as a filtered location that is in dappled shade for much of the day. In general, this species of camellia requires at least 4 hours of sunlight per day, though ideally it should receive 6 to 8 hours of direct sunlight. If too little sunlight is provided, the leaves will tend to be pale and bleached and the camellia may not bloom. On the other hand, in areas w

In [9]:
guide = fetcher.fetch_care_from_perenual("Camellia sasanqua Thunb.")
guide

2025-07-26 20:58:21 [INFO] __main__: Fetching care data for plant: Camellia sasanqua Thunb.
2025-07-26 20:58:21 [INFO] httpx: HTTP Request: GET https://perenual.com/api/species-care-guide-list?key=sk-BY6T687a9e958639511466&q=Camellia%20sasanqua%20Thunb. "HTTP/1.1 200 OK"
2025-07-26 20:58:21 [ERROR] __main__: Unexpected response structure for Camellia sasanqua Thunb.: list index out of range


{}

In [16]:
fetcher = PlantInfoFetcher()
image_files = ["./../../plant_id_test/test_photos/camellia.jpg"]
organs = ["auto"]
result = fetcher.get_plant_info(image_files=image_files, organs=organs)
result

2025-07-26 21:01:28 [INFO] __main__: Initialized PlantInfoFetcher with timeout=10.0
2025-07-26 21:01:28 [INFO] __main__: Identifying plant and retrieving care information


Files to be sent: [('images', <_io.BufferedReader name='./../../plant_id_test/test_photos/camellia.jpg'>)]
Data to be sent: {'organs': ['auto']}


2025-07-26 21:01:31 [INFO] httpx: HTTP Request: POST https://my-api.plantnet.org/v2/identify/all?api-key=2b10uztmWSXBQhAsctxZTu "HTTP/1.1 200 OK"
2025-07-26 21:01:31 [INFO] __main__: Fetching care data for plant: Camellia sasanqua Thunb.
2025-07-26 21:01:32 [INFO] httpx: HTTP Request: GET https://perenual.com/api/species-care-guide-list?key=sk-BY6T687a9e958639511466&q=Camellia%20sasanqua%20Thunb. "HTTP/1.1 200 OK"
2025-07-26 21:01:32 [ERROR] __main__: Unexpected response structure for Camellia sasanqua Thunb.: list index out of range
2025-07-26 21:01:32 [INFO] __main__: Fetching care data for plant: Camellia
2025-07-26 21:01:36 [INFO] httpx: HTTP Request: GET https://perenual.com/api/species-care-guide-list?key=sk-BY6T687a9e958639511466&q=Camellia "HTTP/1.1 200 OK"
2025-07-26 21:01:36 [INFO] __main__: Completed identification and care retrieval


{'score': 0.36081,
 'common_names': ['Camellia', 'Sasanqua camellia', 'Christmas camellia'],
 'scientific_name': 'Camellia sasanqua Thunb.',
 'watering': 'Camellias (Camellia crapnelliana) enjoy an ample amount of water but should never be overwatered. Aim to water your camellia once or twice a week, ultimately depending on how warm or dry your environment is. A good way to tell when it’s time to water is by allowing your soil to become somewhat dry before watering again, but always avoid allowing the soil to dry completely. Never water your plant’s leaves or leave standing water. Camellias also benefit from misting, which can be done every couple of days or as needed.',
 'sunlight': 'Camellia crapnelliana is best grown in a spot that receives full to partial sun, such as a filtered location that is in dappled shade for much of the day. In general, this species of camellia requires at least 4 hours of sunlight per day, though ideally it should receive 6 to 8 hours of direct sunlight. I